# 03_feature_engineering_market_structure.ipynb

This notebook generates **Market Structure** features for BTCUSDT, including swing points, BOS/CHOCH, and Premium/Discount zones.

### Objectives:
1. Load cleaned 5m price data.
2. Compute real-time vs. confirmed swing highs/lows.
3. Detect Break of Structure (BOS).
4. Calculate Premium, Discount, and Equilibrium zones relative to the current dealing range.
5. Save the market structure features to the feature store.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Git Integration - Pull latest code from GitHub
import os
PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
GITHUB_REPO_URL = "https://github.com/umutergul74/TradeBot.git"

print("Pulling latest code from GitHub...")
os.chdir(PROJECT_ROOT)
# Initialize git and pull
!git init -b main
!git remote remove origin 2>/dev/null || true
!git remote add origin {GITHUB_REPO_URL}
!git fetch origin
!git reset --hard origin/main
print("Codebase updated to latest GitHub commit!")

In [ ]:
# Auto-install dependencies if any key package is missing
try:
    import ccxt
    import vectorbt
    import optuna
    import catboost
    import ta
    import shap
except ImportError:
    print("Some dependencies are missing. Installing from requirements.txt...")
    requirements_path = os.path.join(PROJECT_ROOT, "requirements.txt")
    !pip install -q -r {requirements_path}
    print("Installation complete!")

In [ ]:
import os
import sys
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Configurations & Cleaned Data

In [ ]:
from utils.config import load_global_config
from data.parquet_store import ParquetStore

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")
store = ParquetStore(PROJECT_ROOT)

df_5m = store.load_processed_klines(symbol, "5m")
print(f"Loaded 5m clean dataset with shape: {df_5m.shape}")

## 3. Compute Market Structure Features

In [ ]:
from features.market_structure import compute_market_structure

swing_window = config.get("feature", "market_structure", {}).get("swing_window", 5)
df_ms = compute_market_structure(df_5m, window=swing_window)

print("Market Structure features computed:")
print(df_ms[["confirmed_swing_high", "realtime_swing_high", "bullish_bos", "in_discount"]].sum())

## 4. Save Features to Feature Store

In [ ]:
from utils.io_utils import save_parquet

feature_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_market_structure.parquet")
save_parquet(df_ms, feature_path)
print(f"Market structure features saved to {feature_path}")

## Summary of Generated Features

The following features have been successfully added to the feature store:
1. `realtime_swing_high` / `realtime_swing_low` (Leakage-safe swing indicators)
2. `bullish_bos` / `bearish_bos` (Break of Structure signals)
3. `in_discount` / `in_premium` (Current zone relative to dealing range)

**Next Step**: Run [04_feature_engineering_liquidity_ict_smc.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/04_feature_engineering_liquidity_ict_smc.ipynb) to engineer Fair Value Gaps, Order Blocks, and Liquidity Sweeps.

In [ ]:
# Auto-disconnect Colab runtime to save compute units
AUTO_DISCONNECT = False  # Set to True to enable automatic shutdown
if AUTO_DISCONNECT:
    print("Disconnecting Colab runtime...")
    from google.colab import runtime
    runtime.unassign()